# **MODELO DE CLASIFICACIÓN**
# **Bosque Aleatorio (** Random Forest **)**

Este es un modelo de "ensamble", que se puede entender como **el siguiente paso después de Árbol de Decisión**. En lugar de crear un solo árbol (que puede ser propenso a "memorizar" los datos, u overfitting), un **Random Forest** crea cientos de árboles (`n_estimators=100`) y les pide que "voten" por la mejor clasificación.

**Es como pedir la opinión de 100 expertos en lugar de uno solo.** Son más potentes y robustos que un solo árbol. ¡Ventaja! Al igual que los árboles de decisión, los Random Forest no requieren que escalemos los datos.


****
## **Paso 1: Introducción y Librerías**

**Objetivo:** Entrenar un modelo que pueda predecir la especie de un pingüino basándose en sus medidas físicas (largo y profundidad del pico, largo de la aleta y masa corporal). Usaremos un **Random Forest**, como el siguiente paso después .

In [ ]:
# Librerías para cargar el dataset y construir gráficos
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Librerías necesarias para el modelo DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Métricas rendimiento
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

***
## **Paso 2: Cargar y Explorar los Datos**

Cargaremos el dataset penguins usando Seaborn. Es importante revisar los datos para entender su estructura y ver si hay valores faltantes (NaN).
* Cargar el dataset y Mostrar las primeras 5 filas

In [ ]:
dataset = pd.read_csv('https://github.com/estebangonzalezITM/DataScience/raw/main/MaterialdeEstudio/Semana7/titanic_train.csv')
dataset.head()

* Mostrar un resumen (tipos de datos y valores nulos)


In [ ]:
dataset.info()

***
## **Paso 3: Preparación de Datos (Limpieza de datos)**

El método `.info()` mostró que hay valores nulos (NaN) en varias columnas. **Los modelos de scikit-learn no pueden trabajar con datos faltantes**. Para este ejemplo, usaremos la estrategia más simple: eliminar cualquier fila que contenga un valor nulo.

* Eliminamos filas duplicadas

In [ ]:
dataset.drop_duplicates(inplace=True)

* Ahora corregimos los valores faltantes

In [ ]:
print('Valores faltantes:\n',dataset.isnull().sum()[dataset.isnull().sum() > 0])

* Imputar `Age` con la **mediana**

In [ ]:
impute_age = dataset['Age'].median()
dataset.loc[dataset['Age'].isnull(), 'Age'] = impute_age

print('Valores faltantes:\n',dataset.isnull().sum()[dataset.isnull().sum() > 0])

* Imputar `Embarked` con la **moda**

In [ ]:
imp_Embarked = dataset['Embarked'].mode()[0]
dataset.loc[dataset['Embarked'].isnull(), 'Embarked'] = imp_Embarked

print('Valores faltantes:\n',dataset.isnull().sum()[dataset.isnull().sum() > 0])

* Eliminar columnas no útiles: `PassengerId`, `Name`, `Ticket`, `Cabin`.

In [ ]:
dataset.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
dataset.head(3)

* Ahora convertimos variables categóricas a numéricas. `Sex` convertida a `0/1`.

In [ ]:
dataset['Sex'] = dataset['Sex'].map({'male':0, 'female':1})
dataset.head(3)

* La columna puerto de embaque (`Embarked`) es convertida a numerica con One-Hot Encoding.

In [ ]:
dataset = pd.get_dummies(dataset, columns=['Embarked'], prefix='Port')
dataset.head(3)

***
## **Paso 4: Definir Features (X) y Target (y)**

Ahora, separamos nuestros datos en dos partes:

* **Features (`X`):** Las variables de entrada, las "preguntas" que le damos al modelo. Usaremos solo las medidas numéricas.

* **Target (`y`):** La variable de salida, la "respuesta" que queremos predecir (En este caso, las columna `species`).

In [ ]:
dataset.columns

* Lista de columnas que usaremos como features y definimos `X`

In [ ]:
features_list = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Port_C', 'Port_Q', 'Port_S']
X = dataset[features_list]
#X.head()

* Definimos la columna (o lista de columnas) que será nuestro target `y`

In [ ]:
y = dataset['Survived']
#y.head()

***
## **Paso 5: Dividir los datos en Train y Test**

Este es el paso más importante del Machine Learning. El set de datos se divide entre datos para entrenar el modelo y el resto para evaluarlo. Generalmente la relación entre datos de entrenamiento y evaluación es **80-20%** o **70-30%**.

* **Set de Entrenamiento (Train):** Lo usaremos para "entrenar" al modelo (80% de los datos).

*  **Set de Prueba (Test):** Lo guardaremos para "evaluar" al modelo con datos que nunca ha visto (20% de los datos).

* Usamos `random_state=42` para que la división aleatoria sea siempre la misma (reproducible).

In [ ]:
# Dividimos los datos (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total de datos de entrenamiento (train): {len(X_train)}")
print(f"Total de datos de prueba (test): {len(X_test)}")

***
## **Paso 6: Crear y Entrenar el Modelo**

Inicializamos el modelo **Random Forest**. Crearemos un <font color="red">**bosque de 100 árboles**</font> usando `n_estimators=100`.

* Entrenamos el modelo usando el método `.fit()` solo con los datos de entrenamiento.

In [ ]:
# 1. Inicializar el modelo
modelo_rf = RandomForestClassifier(max_depth=3,n_estimators=100, random_state=42)

# 2. Entrenar el modelo
modelo_rf.fit(X_train, y_train)

****
## **Paso 7: Evaluar el Modelo**

## **Paso 7.1: Calculamos la presición (Accuracy)**

Ahora, usamos los datos de prueba (**X_test**) que el modelo nunca vio.

* Hacemos predicciones con los datos de prueba usando `.predict()`

In [ ]:
y_pred = modelo_rf.predict(X_test)

* Comparamos esas predicciones (**y_pred**) con las respuestas correctas(**y_test**). Primero calculamos la **precisión** usando `.accuracy_score()`:

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión (Accuracy) del modelo: {accuracy * 100:.1f}%")

***
## **Paso 7.2: Optimizar los hiperparametros (`GridSearchCV`)**

GridSearchCV es la herramienta que permite que en lugar de probar "a mano" qué número de árboles o qué profundidad funciona mejor, dejas que Python lo haga por ti de forma sistemática.

>### **7.2.1. Definimos el diccionario de parámetros a probar**

In [ ]:
parametros = {
    'n_estimators': [50, 100, 200, 300],        # Número de árboles en el bosque
    'max_depth': [3, 5, 10, None],          # Profundidad máxima de los árboles
}

* **Otros parámetros**
    
```
    'min_samples_split': [2, 5, 10],        # Mínimo de muestras para dividir un nodo
    'criterion': ['gini', 'entropy']        # Función para medir la calidad de la división
```



> ### **7.2.2. Inicializamos el modelo base (sin configurar hiperparámetros)**

In [ ]:
rf = RandomForestClassifier(random_state=42)

> ### **7.2.3. Configuramos GridSearchCV**

* `cv=5` significa que usará Cross-Validation de 5 pliegues, es decir, entrena y prueba cada combinación de parámetros 5 veces.

* `n_jobs=-1` usa todos los núcleos de tu procesador para ir más rápido
* `verbose=1` muestra un resumen al inicio y al final

In [ ]:
grid_search = GridSearchCV(estimator=rf, param_grid=parametros, cv=3, n_jobs=-1, verbose=1)

> ### **7.2.4. Entrenamos buscando la mejor combinación**

In [ ]:
grid_search.fit(X_train, y_train)

> ### **7.2.5. Mostramos los resultados del modelo optimizado**

In [ ]:
print("--- Mejores Hiperparámetros encontrados ---")
print(grid_search.best_params_)

print(f"\nMejor puntuación (Accuracy en Train): {grid_search.best_score_* 100:.1f}%")

mejor_modelo_rf = grid_search.best_estimator_

## **Paso 7.3: Calculamos la matriz de confusión**

 Comparamos esas predicciones (**y_pred**) con las respuestas correctas(**y_test**). Ahora calculamos la **matriz de confusión** usando usando `.confusion_matrix()`:

In [ ]:
print("--- Matriz de Confusión ---")
print("(Las filas son el valor REAL, las columnas la PREDICCIÓN) \n")

labels = mejor_modelo_rf.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

# La mostramos como un DataFrame para que sea más fácil de leer
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

***
## **Paso 8: Importancia de las Características (** Feature Importance **)**

* Obtenemos las importancias y creamos un gráfico de barras para visualizar el resultado más facilmente

In [ ]:
importances = mejor_modelo_rf.feature_importances_
forest_importances = pd.Series(importances, index=features_list)

# Ordenamos de mayor a menor importancia
forest_importances = forest_importances.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=forest_importances.values, y=forest_importances.index)
plt.title("Importancia de las Características en Random Forest")
plt.xlabel("Nivel de Importancia")
plt.ylabel("Característica")
plt.show()